# Pairs Trading: GOOG vs GOOGL
**MFE 412 — Final Project**

**Economic friction:** GOOG and GOOGL are two share classes of Alphabet Inc. with identical
economic claims. Any price divergence is pure noise — no fundamental reason for it to persist.
The friction that sustains the opportunity is **execution cost and inventory risk**: arbitrageurs
must cross two spreads simultaneously, so small dislocations go unexploited for seconds to minutes.
We trade the statistical reversion of the spread back to its long-run equilibrium.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

DATA_DIR = Path('data/raw')
Path('figures').mkdir(exist_ok=True)

TRAIN_DATES = [
    '2024-01-02','2024-01-04','2024-01-08','2024-01-10','2024-01-12',
    '2024-01-16','2024-01-18','2024-01-22','2024-01-24','2024-01-26',
    '2024-02-01','2024-02-05','2024-02-07','2024-02-09','2024-02-12',
    '2024-02-14','2024-02-20','2024-02-22','2024-02-26','2024-02-28',
]
TEST_DATES = [
    '2024-03-04','2024-03-07','2024-03-11','2024-03-14','2024-03-18',
    '2024-04-01','2024-04-04','2024-04-08','2024-04-11','2024-04-15',
]

## 1. Load Data

In [ ]:
def load_mid(symbol: str, dates: list) -> pd.Series:
    """Load MBP-10, compute midprice, resample to 1-minute bars."""
    frames = []
    for d in dates:
        p = DATA_DIR / symbol / 'mbp-10' / f'{d}.parquet'
        if not p.exists():
            print(f'Missing: {p}')
            continue
        df = pd.read_parquet(p).sort_index()
        mid = (df['bid_px_00'] + df['ask_px_00']) / 2
        frames.append(mid)
    if not frames:
        raise FileNotFoundError(f'No data for {symbol}. Run download_pairs.py first.')
    tick = pd.concat(frames)
    return tick.resample('1min').last().dropna()


def load_spread_series(symbol: str, dates: list) -> pd.Series:
    """Return mean bid-ask spread per 1-minute bar (for TC calculation)."""
    frames = []
    for d in dates:
        p = DATA_DIR / symbol / 'mbp-10' / f'{d}.parquet'
        if not p.exists():
            continue
        df = pd.read_parquet(p).sort_index()
        sp = df['ask_px_00'] - df['bid_px_00']
        frames.append(sp)
    return pd.concat(frames).resample('1min').mean().dropna()


print('Loading GOOG...')
goog_mid_train   = load_mid('GOOG',  TRAIN_DATES)
goog_sp_train    = load_spread_series('GOOG',  TRAIN_DATES)
print('Loading GOOGL...')
googl_mid_train  = load_mid('GOOGL', TRAIN_DATES)
googl_sp_train   = load_spread_series('GOOGL', TRAIN_DATES)

# Align on common timestamps
common = goog_mid_train.index.intersection(googl_mid_train.index)
goog  = goog_mid_train.loc[common]
googl = googl_mid_train.loc[common]
goog_sp  = goog_sp_train.reindex(common)
googl_sp = googl_sp_train.reindex(common)

print(f'Common 1-min bars (train): {len(common):,}')
print(f'GOOG  price range: ${goog.min():.2f} – ${goog.max():.2f}')
print(f'GOOGL price range: ${googl.min():.2f} – ${googl.max():.2f}')

## 2. Cointegration Test (Engle-Granger)

In [ ]:
# Engle-Granger cointegration test
score, pvalue, _ = coint(np.log(goog.values), np.log(googl.values))
print(f'Engle-Granger cointegration test (log prices)')
print(f'  Test statistic: {score:.4f}')
print(f'  p-value:        {pvalue:.4f}')
print(f'  Cointegrated:   {"YES" if pvalue < 0.05 else "NO (p > 0.05)"}')

# Estimate hedge ratio beta via OLS on training set
# log(GOOG) = alpha + beta * log(GOOGL) + epsilon
X = add_constant(np.log(googl.values))
y = np.log(goog.values)
res = OLS(y, X).fit()
alpha_hat = res.params[0]
beta_hat  = res.params[1]
print(f'\nHedge ratio (beta): {beta_hat:.6f}')
print(f'Intercept (alpha):  {alpha_hat:.6f}')
print(f'R²: {res.rsquared:.6f}')

# ADF test on the residuals (spread must be stationary)
spread_log = y - (alpha_hat + beta_hat * np.log(googl.values))
adf_stat, adf_p, _, _, crit, _ = adfuller(spread_log, autolag='AIC')
print(f'\nADF test on spread residuals')
print(f'  ADF statistic: {adf_stat:.4f}')
print(f'  p-value:       {adf_p:.4f}')
print(f'  Stationary:    {"YES" if adf_p < 0.05 else "NO"}')
print(f'  Critical values: {crit}')

## 3. Visualize the Spread

In [ ]:
spread_train = pd.Series(spread_log, index=common, name='spread')

fig, axes = plt.subplots(3, 1, figsize=(13, 8))

# Price ratio
ratio = goog / googl
axes[0].plot(ratio.values, lw=0.8, color='steelblue')
axes[0].axhline(ratio.mean(), color='red', lw=1, ls='--', label=f'Mean = {ratio.mean():.4f}')
axes[0].set_ylabel('GOOG / GOOGL price ratio')
axes[0].set_title('GOOG vs GOOGL — Training Set')
axes[0].legend()

# Log spread (residual)
axes[1].plot(spread_train.values, lw=0.8, color='darkorange')
axes[1].axhline(0, color='black', lw=0.5, ls='--')
axes[1].set_ylabel('Log-price spread residual')

# Autocorrelation of spread (shows mean-reversion speed)
lags = range(1, 31)
autocorrs = [spread_train.autocorr(lag=l) for l in lags]
axes[2].bar(list(lags), autocorrs, color='steelblue', alpha=0.8)
axes[2].axhline(0, color='black', lw=0.5)
axes[2].set_xlabel('Lag (minutes)')
axes[2].set_ylabel('Autocorrelation')

plt.tight_layout()
plt.savefig('figures/pairs_spread_overview.png', dpi=150)
plt.show()

half_life_est = -np.log(2) / np.log(abs(spread_train.autocorr(1)))
print(f'Estimated mean-reversion half-life: {half_life_est:.1f} minutes')

## 4. Signal: Rolling Z-Score

In [ ]:
# Parameters — calibrated on training data, frozen before test
ROLL_WINDOW  = 120   # 2-hour rolling window for mean/std estimation
ENTRY_Z      = 2.0   # enter when |z| > ENTRY_Z
EXIT_Z       = 0.5   # exit when |z| < EXIT_Z (near mean)
STOP_Z       = 3.5   # stop-loss: spread widening further
MAX_HOLD     = 30    # maximum holding time in minutes
NOTIONAL     = 10_000  # $ per leg


def compute_zscore(spread: pd.Series, window: int = ROLL_WINDOW) -> pd.Series:
    """Rolling z-score using only past data (no look-ahead)."""
    roll_mean = spread.shift(1).rolling(window, min_periods=window // 2).mean()
    roll_std  = spread.shift(1).rolling(window, min_periods=window // 2).std()
    return ((spread - roll_mean) / roll_std.replace(0, np.nan)).rename('zscore')


zscore_train = compute_zscore(spread_train)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(spread_train.values, lw=0.7, color='steelblue')
axes[0].set_ylabel('Spread')
axes[1].plot(zscore_train.values, lw=0.7, color='darkorange')
axes[1].axhline(0, color='black', lw=0.5)
axes[1].axhline( ENTRY_Z, color='green',  lw=1, ls='--', label=f'+{ENTRY_Z} entry')
axes[1].axhline(-ENTRY_Z, color='green',  lw=1, ls='--', label=f'-{ENTRY_Z} entry')
axes[1].axhline( STOP_Z,  color='red',    lw=1, ls=':',  label=f'+{STOP_Z} stop')
axes[1].axhline(-STOP_Z,  color='red',    lw=1, ls=':')
axes[1].set_ylabel('Z-score')
axes[1].set_xlabel('Bar index')
axes[1].legend(fontsize=8)
plt.suptitle('Spread Z-Score — Training Set', fontsize=12)
plt.tight_layout()
plt.savefig('figures/pairs_zscore.png', dpi=150)
plt.show()

print(f'Z-score > +{ENTRY_Z}: {(zscore_train >  ENTRY_Z).sum()} bars')
print(f'Z-score < -{ENTRY_Z}: {(zscore_train < -ENTRY_Z).sum()} bars')

## 5. Backtest

In [ ]:
def backtest_pairs(
    goog: pd.Series, googl: pd.Series,
    goog_sp: pd.Series, googl_sp: pd.Series,
    beta: float, alpha: float,
    notional: float = NOTIONAL,
    entry_z: float = ENTRY_Z,
    exit_z: float  = EXIT_Z,
    stop_z: float  = STOP_Z,
    max_hold: int  = MAX_HOLD,
    roll_window: int = ROLL_WINDOW,
) -> pd.DataFrame:
    """
    Event-driven pairs backtest.

    Spread = log(GOOG) - alpha - beta * log(GOOGL)
    Z      = (spread - rolling_mean) / rolling_std

    Long spread  (z < -entry_z): BUY  GOOG, SELL GOOGL
    Short spread (z >  entry_z): SELL GOOG, BUY  GOOGL

    TC = half_spread per leg at entry and exit.
    Position sized to $notional per leg.
    """
    spread = np.log(goog) - alpha - beta * np.log(googl)
    zscore = compute_zscore(spread, roll_window)

    prices_g  = goog.values
    prices_gl = googl.values
    sp_g  = goog_sp.reindex(goog.index).values
    sp_gl = googl_sp.reindex(googl.index).values
    z     = zscore.values
    n     = len(z)

    trades = []
    in_trade = False
    direction = entry_bar = 0  # +1 = long spread, -1 = short spread

    for i in range(1, n - 1):
        if np.isnan(z[i]):
            continue

        if in_trade:
            bars_held = i - entry_bar
            z_cur = z[i]
            # Exit conditions
            mean_rev = (direction ==  1 and z_cur >  -exit_z) or \
                       (direction == -1 and z_cur <   exit_z)
            stop     = (direction ==  1 and z_cur < -stop_z) or \
                       (direction == -1 and z_cur >  stop_z)
            timeout  = bars_held >= max_hold

            if mean_rev or stop or timeout:
                # Calculate PnL
                sh_g  = notional / prices_g[entry_bar]
                sh_gl = notional / prices_gl[entry_bar]

                # Leg returns
                ret_g  = (prices_g[i]  - prices_g[entry_bar])  * sh_g  * direction
                ret_gl = (prices_gl[i] - prices_gl[entry_bar]) * sh_gl * (-direction)
                gross  = ret_g + ret_gl

                # TC: half-spread × shares, both legs, entry + exit
                tc = (
                    (sp_g[entry_bar] / 2  + sp_g[i] / 2)  * sh_g  +
                    (sp_gl[entry_bar] / 2 + sp_gl[i] / 2) * sh_gl
                )

                exit_reason = 'mean_rev' if mean_rev else ('stop' if stop else 'timeout')
                trades.append({
                    'entry_time':  goog.index[entry_bar],
                    'exit_time':   goog.index[i],
                    'direction':   direction,
                    'z_entry':     z[entry_bar],
                    'z_exit':      z_cur,
                    'bars_held':   bars_held,
                    'gross_pnl':   gross,
                    'tc':          tc,
                    'net_pnl':     gross - tc,
                    'exit_reason': exit_reason,
                })
                in_trade = False

        else:
            # Entry on next bar after signal (1-bar delay = 1 min)
            if z[i - 1] < -entry_z:
                direction = 1    # long spread: buy GOOG, sell GOOGL
                entry_bar = i
                in_trade  = True
            elif z[i - 1] > entry_z:
                direction = -1   # short spread: sell GOOG, buy GOOGL
                entry_bar = i
                in_trade  = True

    return pd.DataFrame(trades)


trades_train = backtest_pairs(
    goog, googl, goog_sp, googl_sp,
    beta=beta_hat, alpha=alpha_hat
)
print(f'Trades: {len(trades_train)}')
if len(trades_train):
    print(trades_train['exit_reason'].value_counts().to_string())
    print(f'Hit rate:   {(trades_train.net_pnl > 0).mean():.1%}')
    print(f'Total PnL:  ${trades_train.net_pnl.sum():.2f}')
    print(f'Mean TC:    ${trades_train.tc.mean():.4f}')
    print(f'Mean hold:  {trades_train.bars_held.mean():.1f} min')

## 6. Performance Metrics

In [ ]:
def compute_metrics(trades: pd.DataFrame, label: str = '') -> dict:
    if trades.empty:
        print('No trades.')
        return {}
    daily = trades.set_index('entry_time').resample('D')['net_pnl'].sum()
    daily = daily[daily != 0]
    ann_ret = daily.mean() * 252
    ann_vol = daily.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan
    cum = trades['net_pnl'].cumsum()
    mdd = (cum - cum.cummax()).min()
    m = {
        'label':       label,
        'n_trades':    len(trades),
        'total_pnl':   round(trades.net_pnl.sum(), 2),
        'ann_return':  round(ann_ret, 2),
        'ann_vol':     round(ann_vol, 2),
        'sharpe':      round(sharpe, 3),
        'max_drawdown':round(mdd, 2),
        'hit_rate':    round((trades.net_pnl > 0).mean(), 3),
        'avg_hold_min':round(trades.bars_held.mean(), 1),
        'pct_mean_rev':round((trades.exit_reason == 'mean_rev').mean(), 3),
    }
    for k, v in m.items():
        print(f'  {k:20s}: {v}')
    return m

print('=== TRAINING ===')
m_train = compute_metrics(trades_train, 'train')

## 7. Plots

In [ ]:
def plot_results(trades: pd.DataFrame, title: str, fname: str):
    if trades.empty:
        print('No trades to plot.')
        return
    fig, axes = plt.subplots(3, 1, figsize=(12, 9))

    cum = trades['net_pnl'].cumsum()
    axes[0].plot(cum.values, lw=1.5, color='steelblue')
    axes[0].axhline(0, color='black', lw=0.5, ls='--')
    axes[0].set_ylabel('Cumulative PnL ($)')
    axes[0].set_title(title)

    # Rolling Sharpe (20-trade window)
    w = min(20, len(trades))
    rs = (trades['net_pnl'].rolling(w).mean() /
          trades['net_pnl'].rolling(w).std().replace(0, np.nan)) * np.sqrt(252)
    axes[1].plot(rs.values, lw=1, color='darkorange')
    axes[1].axhline(0, color='black', lw=0.5, ls='--')
    axes[1].set_ylabel(f'Rolling Sharpe ({w}-trade)')

    colors = ['steelblue' if r == 'mean_rev' else 'tomato' for r in trades['exit_reason']]
    axes[2].bar(range(len(trades)), trades['net_pnl'].values, color=colors, alpha=0.7, width=0.8)
    axes[2].axhline(0, color='black', lw=0.5)
    axes[2].set_ylabel('Per-trade PnL ($)')
    axes[2].set_xlabel('Trade #  (blue=mean_rev, red=stop/timeout)')

    plt.tight_layout()
    plt.savefig(f'figures/{fname}', dpi=150)
    plt.show()


plot_results(trades_train, 'Pairs Trading GOOG/GOOGL — Training', 'pairs_train_pnl.png')

## 8. Beta vs SPX (market-neutrality check)

In [ ]:
# Pairs strategy is long one leg, short the other → should have beta ≈ 0
# Use GOOG daily return as SPX proxy (or load SPY if available)

def beta_check(trades: pd.DataFrame, market_mid: pd.Series) -> None:
    mkt_daily  = market_mid.resample('D').last().pct_change().dropna() * 100
    strat_daily = trades.set_index('entry_time').resample('D')['net_pnl'].sum()
    strat_daily = strat_daily[strat_daily != 0]
    common = strat_daily.index.intersection(mkt_daily.index)
    if len(common) < 5:
        print('Not enough overlapping days.')
        return
    x = mkt_daily.loc[common].values
    y = strat_daily.loc[common].values
    slope, intercept, r, p, _ = stats.linregress(x, y)
    print(f'Beta vs GOOG (SPX proxy):  {slope:.4f}  (p={p:.3f})')
    print(f'Daily alpha:               ${intercept:.4f}  → ${intercept*252:.2f}/yr')
    print(f'R²:                        {r**2:.4f}')
    print('→ Beta ≈ 0 confirms market-neutral structure.')


beta_check(trades_train, goog)

## 9. Out-of-Sample Test

In [ ]:
print('Loading test data...')
goog_test   = load_mid('GOOG',  TEST_DATES)
googl_test  = load_mid('GOOGL', TEST_DATES)
goog_sp_test  = load_spread_series('GOOG',  TEST_DATES)
googl_sp_test = load_spread_series('GOOGL', TEST_DATES)

common_test = goog_test.index.intersection(googl_test.index)
goog_test   = goog_test.loc[common_test]
googl_test  = googl_test.loc[common_test]

# Use TRAINING beta and alpha — no re-estimation on test data
trades_test = backtest_pairs(
    goog_test, googl_test,
    goog_sp_test, googl_sp_test,
    beta=beta_hat, alpha=alpha_hat   # frozen from training
)

print('\n=== TEST (OOS) ===')
m_test = compute_metrics(trades_test, 'test')
plot_results(trades_test, 'Pairs Trading GOOG/GOOGL — Test (OOS)', 'pairs_test_pnl.png')

## 10. Parameter Sensitivity (training only)

In [ ]:
results = []
for ez in [1.5, 2.0, 2.5]:
    for win in [60, 120, 240]:
        t = backtest_pairs(goog, googl, goog_sp, googl_sp,
                           beta=beta_hat, alpha=alpha_hat,
                           entry_z=ez, roll_window=win)
        if not t.empty:
            daily = t.set_index('entry_time').resample('D')['net_pnl'].sum()
            daily = daily[daily != 0]
            sharpe = (daily.mean() * 252) / (daily.std() * np.sqrt(252) + 1e-9)
            results.append({'entry_z': ez, 'window': win,
                             'n_trades': len(t),
                             'total_pnl': round(t.net_pnl.sum(), 2),
                             'sharpe': round(sharpe, 3)})

sens = pd.DataFrame(results)
print(sens.pivot_table(index='entry_z', columns='window', values='sharpe'))
print()
print(sens.pivot_table(index='entry_z', columns='window', values='total_pnl'))

## 11. Summary

In [ ]:
summary = pd.DataFrame([m_train, m_test]).set_index('label')
summary.T